# TrialScreen: full pipeline on Kaggle

**Before running**, in the right-hand panel:
1. Accelerator: **GPU T4 x1** (T4 x2 also works; one GPU is used).
2. Internet: **On** (needs a phone-verified Kaggle account).
3. Add-ons → Secrets: add **HF_TOKEN** (a Hugging Face *write* token) and tick it for this notebook.

Then click **Save Version → Save & Run All (Commit)**. It runs on Kaggle's servers (about 1.5 to 2 hours),
so closing the browser does not stop it. When it finishes, download `trialscreen_outputs.zip`
from the version's **Output** tab.

`!` commands that fail do not stop the run, so the final cell always packages whatever was produced.

In [ ]:
# 0. Environment checks: GPU and internet
import os, subprocess, sys, time
WORK = "/kaggle/working"
net = subprocess.run(["curl", "-sI", "--max-time", "15", "https://github.com"], capture_output=True, text=True).stdout.split("\n")[0]
assert net.startswith("HTTP"), "No internet: Settings > Internet On (phone-verified account needed)."
import torch
assert torch.cuda.is_available(), "No GPU: Settings > Accelerator > GPU T4."
print("internet OK |", torch.cuda.get_device_name(0), "| python", sys.version.split()[0], "| torch", torch.__version__)
T0 = time.time()

In [ ]:
# 1. Get the latest code from GitHub
REPO = "https://github.com/eman-huda/trialscreen.git"
ROOT = f"{WORK}/trialscreen"
subprocess.run(["rm", "-rf", ROOT])
subprocess.run(["git", "clone", "--depth", "1", REPO, ROOT], check=True)
os.chdir(ROOT)
print(subprocess.run(["git", "log", "-1", "--format=%h %s (%ci)"], capture_output=True, text=True).stdout)
# make sure the logits/memory fix is in the repo before spending GPU time
src = open("src/trialscreen/decoding.py").read() + open("src/trialscreen/modeling.py").read()
assert "logits_to_keep" in src and "get_device_capability" in src, ("The repo is missing the latest fix (trialscreen_update2.zip). "
                                 "Unzip it into the repo in Codespaces, commit and push, then run again.")
print(sorted(os.listdir(".")))

In [ ]:
# 2. Install pinned libraries
!pip install -q "transformers==5.22.2" "peft==0.21.2" "accelerate==1.15.0" scikit-learn matplotlib pytest
!pip install -q -e .
# Kaggle ships torchao 0.10, which peft rejects when creating LoRA layers; the project does not use torchao
!pip uninstall -y -q torchao
!python -c "import transformers, peft; print('transformers', transformers.__version__, '| peft', peft.__version__)"
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Data: fixed 37/8/8 patient split (seed 42), leakage checks, unit tests

In [ ]:
!python -u scripts/01_prepare_data.py
!python -m pytest -q tests 2>&1 | tail -3 | tee results/pytest_gpu_env.txt

## Task 1: LoRA fine-tuning (train split; best epoch chosen by validation macro-F1)

In [ ]:
!python -u scripts/02_train_lora.py
!ls -la checkpoints/qwen2.5-1.5b-trialscreen-lora/best_adapter

## Upload the adapter to Hugging Face immediately after training
If the session is lost later, the trained adapter is safe on the Hub and the app can use it.

In [ ]:
HF_REPO = "eman-huda/qwen2.5-1.5b-trialscreen-lora"   # change if your Hugging Face username is different
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    r = subprocess.run([sys.executable, "scripts/08_push_adapter.py", "--repo", HF_REPO], capture_output=True, text=True)
    print(r.stdout[-2000:], r.stderr[-2000:])
except Exception as e:
    print("Adapter NOT uploaded:", e, "\nIt is still inside trialscreen_outputs.zip; upload it later with scripts/08_push_adapter.py.")

## Before/after evaluation (validation, then the test split once)

In [ ]:
!python -u scripts/03_evaluate.py --split validation

In [ ]:
!python -u scripts/03_evaluate.py --split test --final

## Task 3: KV-cache benchmark (3 input x 3 output lengths, warm-up, 3 repeats; fp32 recheck of any fp16 mismatch)

In [ ]:
!python -u scripts/04_bench_kv.py

## Task 4: tiled online-softmax attention

In [ ]:
!python -u scripts/05_attention.py

## Task 5: greedy speculative decoding (draft Qwen2.5-0.5B-Instruct)

In [ ]:
!python -u scripts/06_bench_speculative.py

## Report tables, figures and error-analysis worksheet

In [ ]:
!python -u scripts/07_make_report_assets.py
!ls report/figures report/tables results

## Package all outputs

In [ ]:
out = f"{WORK}/trialscreen_outputs.zip"
!cd {ROOT} && zip -qr {out} results report data/splits checkpoints/qwen2.5-1.5b-trialscreen-lora/best_adapter checkpoints/qwen2.5-1.5b-trialscreen-lora/*.json 2>/dev/null; ls -lh {out}
print(f"total time {(time.time() - T0) / 60:.0f} min")
# keep only the zip and the repo in the output panel
!rm -rf {WORK}/trialscreen/data/raw